# Evidence-aware ranking: executable walkthrough

A synthetic demonstration of feature construction, candidate scoring, and grouped evaluation. These outputs are not empirical research findings. Run from the repository root or notebooks directory after `pip install -e .`.

In [1]:
from pathlib import Path
import json
import pandas as pd
from pec_fuse import *
ROOT = Path.cwd()
if not (ROOT / "examples").exists():
    ROOT = ROOT.parent
inputs = pd.read_csv(ROOT / "examples/synthetic/candidate_inputs.csv")
inputs.head()

  query_id user_id candidate_id  split  ...  is_breaking  C_max  C_top3  C_attention
0       q1      u1           a1  train  ...            0   0.86    0.78         0.74
1       q1      u1           a2  train  ...            1   0.31    0.26         0.24
2       q1      u1           a3  train  ...            0   0.48    0.39         0.36
3       q1      u1           a4  train  ...            0   0.18    0.13         0.12
4       q1      u1           a5  train  ...            0   0.41    0.34         0.32

[5 rows x 21 columns]

## 1. Preserve the input sources
Stored preference, retained recommendation, and click-history summaries remain separate fields. The builder validates retained-rank consistency and derives cross-source interactions. Upstream semantic summaries must not access target labels.

In [2]:
rows = build_feature_frame(inputs)
rows[["query_id", "candidate_id", *FEATURES_25]].head()

  query_id candidate_id  ...  C_max_x_E  P_semantic_x_C_attention
0       q1           a1  ...      0.000                    0.6068
1       q1           a2  ...      0.620                    0.0432
2       q1           a3  ...      0.720                    0.1404
3       q1           a4  ...      0.000                    0.0108
4       q1           a5  ...      0.492                    0.0992

[5 rows x 27 columns]

## 2. Compare an operational baseline with the complete record
Both models use the fixed configuration supplied with the source implementation. This notebook does not tune on the test fixture.

In [3]:
train = rows[rows["split"].eq("train")].copy()
test = rows[rows["split"].eq("test")].copy()
cfg = json.loads((ROOT / "configs/extra_trees.json").read_text())["recent_window"]
control = PECFuseRanker(features=STRONG_CONTROL_FEATURES, **cfg["strong_control"]).fit(train)
full = PECFuseRanker(features=FEATURES_25, **cfg["pec_fuse"]).fit(train)
ranked = rank_within_query(test, full.score(test))
ranked[["query_id", "candidate_id", "label", "score", "rank"]]

  query_id candidate_id  label     score  rank
0       q3           c1      1  0.983333     1
1       q3           c2      0  0.002381     2
2       q3           c3      0  0.002381     3
3       q3           c4      0  0.000000     4
4       q3           c5      0  0.000000     5

## 3. Evaluate within queries, then resample users
All queries receive equal weight in the mean. Cluster resampling preserves dependence among queries from the same user. On this tiny fixture the interval is illustrative only; ties preserve input order.

In [4]:
baseline = query_metrics(rank_within_query(test, control.score(test)))
challenger = query_metrics(ranked)
print("Baseline:", aggregate_metrics(baseline))
print("Complete:", aggregate_metrics(challenger))
paired_user_cluster_bootstrap(challenger, baseline, iterations=200, random_state=42)

Baseline: {'ndcg@10': 1.0, 'recall@10': 1.0, 'mrr': 1.0}
Complete: {'ndcg@10': 1.0, 'recall@10': 1.0, 'mrr': 1.0}


{'queries': 1, 'users': 1, 'mean_delta': 0.0, 'ci_low': 0.0, 'ci_high': 0.0, 'p_delta_le_zero': 1.0}

## Interpretation
A runnable scoring pipeline does not establish attention, causal impact, or superior recommendation quality. The optional MIND adapter adds a public-data example with explicit proxy assumptions; see the README before interpreting its metrics.